# Ollama on Colab, exposed with ngrok

Runs an Ollama model on a Colab GPU and prints a public URL for the Stock Research Assistant.

1. **Runtime → Change runtime type → T4 GPU** (or any GPU).
2. Create a free ngrok account and copy your authtoken from the ngrok dashboard.
3. In Colab open **Secrets** (key icon on the left), add `NGROK_AUTHTOKEN` and allow this notebook to read it.
4. **Runtime → Run all**. The last cell prints the lines to put in your `.env`.

Notes:
- Anyone with the URL can use your model. Keep it private and stop the runtime when you are done.
- The URL changes every time the runtime restarts; update `LLM_API_BASE` each time.
- Colab ends idle or long sessions. Set `LLM_FALLBACK_*` in `.env` so the app keeps working.
- Colab's free tier restricts remote proxies; check the Colab FAQ and ngrok terms for your plan before relying on this for a demo.

In [ ]:
# Model with native tool calling. qwen3:8b needs about 6 GB of GPU memory.
MODEL = "qwen3:8b"
OLLAMA_PORT = 11434

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: switch the runtime to a GPU"
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pyngrok

In [ ]:
import os
import subprocess
import time
import urllib.request

# Bind to 0.0.0.0: Ollama then accepts any Host header, which requests arriving through
# ngrok carry. The Colab VM is not reachable from outside; only the ngrok tunnel exposes it.
env = dict(os.environ, OLLAMA_HOST=f"0.0.0.0:{OLLAMA_PORT}", OLLAMA_KEEP_ALIVE="30m")
ollama_log = open("ollama.log", "w")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/version", timeout=2)
        print("Ollama is running")
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see ollama.log")

In [ ]:
!ollama pull {MODEL}
!ollama list

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
# Also rewrite the Host header, in case Ollama is ever bound to localhost only.
tunnel = ngrok.connect(OLLAMA_PORT, "http", host_header=f"localhost:{OLLAMA_PORT}")
PUBLIC_URL = tunnel.public_url
print("Public URL:", PUBLIC_URL)

In [ ]:
import json
import urllib.request

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "What is the price of AAPL?"}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "get_quote",
            "description": "Latest price for a NASDAQ stock",
            "parameters": {
                "type": "object",
                "properties": {"symbol": {"type": "string"}},
                "required": ["symbol"],
            },
        },
    }],
}
request = urllib.request.Request(
    f"{PUBLIC_URL}/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json", "ngrok-skip-browser-warning": "true"},
)
with urllib.request.urlopen(request, timeout=300) as response:
    message = json.load(response)["choices"][0]["message"]
print("Tool calls through the public URL:", message.get("tool_calls"))

print("\nPut these lines in your .env:\n")
print("LLM_PROVIDER=ollama")
print(f"LLM_MODEL={MODEL}")
print(f"LLM_API_BASE={PUBLIC_URL}")

Keep this tab open while you use the app. To stop, run the cell below or end the runtime.

In [ ]:
ngrok.kill()
ollama.terminate()
print("Stopped")